# Renal planner — validation on public data (Kaggle / Colab)

Runs the automatic pipeline (TotalSegmentator + AbdomenAtlasNet-Tumors) on public KiTS23 cases
with expert labels and reports:

* **segmentation accuracy**: renal-mass and tumour Dice, lesion detection by size;
* **plan agreement**: planner on *expert* labels vs on *automatic* labels (R.E.N.A.L.
  components, diameter, nearness, parenchyma kept).

No patient data is involved.

**Before running (Kaggle):** *Settings → Accelerator → GPU* (T4 ×2 or P100) and
*Settings → Internet → On* (needs a phone-verified account). For a long run use
**Save Version → Save & Run All**: it runs in the background, so you can close the browser.

> ⚠️ KiTS23 is one of the datasets AbdomenAtlasNet was trained on, so KiTS tumour scores are
> **optimistic**. For an unbiased number use held-out data (KiPA22 section below, or your own).
> Research software — not a clinical validation.

In [ ]:
import sys

print(sys.version)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Code and dependencies (about 5 min)
%cd /tmp
!rm -rf renal-ct-explorer && git clone -q --depth 1 -b planning-engine https://github.com/ahmederaky2-cmyk/renal-ct-explorer.git
%cd /tmp/renal-ct-explorer
!pip install -q -r requirements-patient.txt

In [ ]:
# Models: AbdomenAtlasNet-Tumors (CC BY-NC 4.0, ~2.3 GB).
# TotalSegmentator downloads its own weights.
import os
import zipfile

from huggingface_hub import hf_hub_download

MODELS = "/tmp/models"
tumor_dir = f"{MODELS}/AbdomenAtlasNetTumors"
if not os.path.isdir(tumor_dir):
    archive = hf_hub_download("AbdomenAtlas/AbdomenAtlasNet", "AbdomenAtlasNetTumors.zip")
    zipfile.ZipFile(archive).extractall(tumor_dir)

os.environ.update(
    RENAL_TOTALSEG_BACKEND="pytorch",
    RENAL_DEVICE="gpu",
    RENAL_ATLASNET_TUMOR_DIR=tumor_dir,
    RENAL_ATLASNET_FOLDS="0",       # one fold is ~5x faster; use "all" for the full ensemble
    RENAL_RUN_KIDNEY_CYSTS="1",
)

## Settings

`LIMIT` cases from `validation/scoring_cases.txt` (the same 40 cases as your scoring sheet).
Allow roughly 5–10 minutes per case on a T4. Results are written after every case.

In [ ]:
import shutil

LIMIT = 20
ON_KAGGLE = os.path.isdir("/kaggle")
OUT = "/kaggle/working/validation_kits23" if ON_KAGGLE else "/content/validation_kits23"

# Continue a previous session: attach the previous notebook version's output as input
# and set its folder here (e.g. "/kaggle/input/<notebook-name>/validation_kits23").
PREVIOUS = ""
if PREVIOUS:
    shutil.copytree(PREVIOUS, OUT, dirs_exist_ok=True)

In [ ]:
!python validation/run_validation.py --dataset kits23 \
    --cases-file validation/scoring_cases.txt --limit {LIMIT} \
    --work /tmp/work --out {OUT} --time-limit-hours 10.5

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown(open(f"{OUT}/report.md").read()))
results = pd.read_csv(f"{OUT}/results.csv")
columns = [c for c in ["case", "renal_mass_dice", "tumor_dice", "found", "true_lesions",
           "false_positive_lesions", "expert_renal_total", "auto_renal_total",
           "expert_diameter_mm", "auto_diameter_mm", "error"] if c in results]
results[columns]

## Compare with your own R.E.N.A.L. scores

Fill in the scoring sheet **blind** (before looking at these results), export it as CSV,
add it to this notebook (*Add Input → Upload*), and set the path:

In [ ]:
MANUAL = ""   # e.g. "/kaggle/input/my-renal-scores/scores.csv"
if MANUAL:
    !python validation/compare_scores.py --manual {MANUAL} --planner {OUT}/results.csv --out {OUT}

## Optional: held-out check on KiPA22 (unbiased for the tumour model)

KiPA22 must not be redistributed, so add it as a **private** Kaggle dataset in the layout
`case_xxxxx/imaging.nii.gz + segmentation.nii.gz`. KiPA22 is cropped CTA, so whole-body
models can struggle at the crop edges; read the report with that in mind.

In [ ]:
KIPA_ROOT = ""   # e.g. "/kaggle/input/kipa22-private/train"
if KIPA_ROOT:
    !python validation/run_validation.py --dataset kipa22 --root {KIPA_ROOT} --limit 30 \
        --work /tmp/work_kipa --out /kaggle/working/validation_kipa22 --time-limit-hours 10.5